In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import auc, roc_curve

In [ ]:
# 讀取訓練資料與測試資料
df_train = pd.read_csv('./cs-training.csv')
df_test = pd.read_csv('./cs-test.csv')

In [ ]:
# 檢視訓練資料的基本資訊
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

In [ ]:
# 檢視測試資料的基本資訊
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 101503 entries, 0 to 101502
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            101503 non-null  int64  
 1   SeriousDlqin2yrs                      0 non-null       float64
 2   RevolvingUtilizationOfUnsecuredLines  101503 non-null  float64
 3   age                                   101503 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  101503 non-null  int64  
 5   DebtRatio                             101503 non-null  float64
 6   MonthlyIncome                         81400 non-null   float64
 7   NumberOfOpenCreditLinesAndLoans       101503 non-null  int64  
 8   NumberOfTimes90DaysLate               101503 non-null  int64  
 9   NumberRealEstateLoansOrLines          101503 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  101503 non-null  int64  
 11  NumberOfDep

In [ ]:
# 取得訓練資料的特徵與目標變數
X = df_train.iloc[:, 2:12]
y = df_train["SeriousDlqin2yrs"]

# 將資料分為訓練集與測試集
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

# 特徵縮放 (標準化)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
# 選擇模型
rf = RandomForestClassifier(random_state=0)

# 定義超參數網格
param_grid = {
    "n_estimators": np.linspace(1, 500, 2, dtype=int),
    "max_depth": [None] + list(np.linspace(1, 500, 2, dtype=int)),
    "min_samples_split": np.linspace(2, 100, 2, dtype=int)
}

# 使用 GridSearchCV 進行超參數調整
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=3,
    n_jobs=4,
    verbose=3
)

# 執行 GridSearchCV
grid_search.fit(X_train_scaled, y_train)
print("Best Hyperparameters:", grid_search.best_params_)

# 使用最佳模型進行預測
y_pred = grid_search.predict(X_test_scaled)

# 計算 AUC-ROC
score = roc_auc_score(y_test, y_pred)
print("AUC-ROC:", score)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Hyperparameters: {'max_depth': None, 'min_samples_split': np.int64(100), 'n_estimators': np.int64(500)}
AUC-ROC: 0.5825150910899212


In [8]:
# 取得 df_test 的特徵
X_ = df_test.iloc[:, 2:12]

# 標準化
X_scaled = scaler.transform(X_)

# 預測機率
y_pred_proba = grid_search.predict_proba(X_scaled)[:, 1]

# sampleEntry.csv 的欄位有 Id, Probability
submission = pd.DataFrame({
    "Id": df_test.index + 1,
    "Probability": y_pred_proba
})

# 儲存為 submission.csv
submission.to_csv("submission.csv", index=False)